# 🤖 Predictive Machine Learning & Student Segmentation
- **Model 1**: Early Coursework to Final Grand Total Regression
- **Model 2**: At-Risk Student Classification (Failure Early Warning)
- **Model 3**: Unsupervised Student Persona Segmentation (K-Means)


In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../src')
from ml.feature_engineering import load_and_engineer_features
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, classification_report
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = load_and_engineer_features()
print("Features engineered successfully.")


### 1. Regression: Predict Final Total Marks from Continuous Assessment

In [ ]:
features = [
    "OE_internal", "CT_internal", "DBMS_internal", "OS_internal", "FTS_internal",
    "MINIPROJ_term_work", "MINIPROJ_oral", "DBMS_LAB_term_work", "DBMS_LAB_oral",
    "OS_LAB_term_work", "OS_LAB_oral", "TC_LAB_term_work", "TC_LAB_oral",
    "BMD_term_work", "DT_term_work"
]

X = df[features].fillna(0)
y = df['overall_total']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred = rf_reg.predict(X_test)

print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} marks")


### 2. Classification: Early Identification of At-Risk Students

In [ ]:
y_cls = df['is_at_risk']
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X, y_cls, test_size=0.25, random_state=42, stratify=y_cls)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_tr_c, y_tr_c)
y_pred_c = rf_clf.predict(X_te_c)

print(classification_report(y_te_c, y_pred_c, target_names=['Safe', 'At-Risk']))


### 3. Student Segmentation (K-Means Clustering)

In [ ]:
cluster_cols = ['theory_avg_pct', 'lab_avg_pct', 'internal_total_score', 'external_total_score']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_cols].fillna(0))

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(df['theory_avg_pct'], df['lab_avg_pct'], c=df['cluster'], cmap='viridis', s=60, alpha=0.85, edgecolors='k')
plt.xlabel('Theory Average (%)', fontsize=12)
plt.ylabel('Lab & Practical Average (%)', fontsize=12)
plt.title('Student Academic Clusters (Theory vs Practical Performance)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster ID')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
